# Find bounding box for data

#### Project bounding box for all data:

| boundary | value | 
|----------|-------|
| West longitude | -85.94712712079293 |
| East longitide | -85.3443621648922 |
| South_latitude | 37.99712528351634 |
| North_latitude | 38.38023822809115 |

#### Span

| | difference | approx span | 
|-|------------|-------------|
| ∆ longitude | 0.6027649559007244 | 30 miles |
| ∆ latitude | 0.3831129445748118 | 26 miles |

\* Code for finding the boundaries begins below.

In [1]:

import json

from os import path

import pandas as pd
import numpy as np

from pyproj import CRS
from pyproj.transformer import Transformer 

HOME = "~/code/county_coverage/"


In [2]:
# general functions

def get_data(filepath) -> dict:
    fp = path.join(path.expanduser(HOME), filepath)
    with open(fp, 'r') as data:
        json_data = json.load(data)
    return json_data


def sort_long_lat(points:list) -> pd.Series:
    """Create bounding box from a list of (longitude, latitude) points."""
    longitudes = set()
    latitudes = set()
    for long, lat in points:
        longitudes.add(long)
        latitudes.add(lat)

    return {"west_longitude": min(longitudes), "east_longitude": max(longitudes),
            "south_latitude": min(latitudes), "north_latitude": max(latitudes)}


Source data for county boundary is a GeoJSON file that looks like this:

```json
{
"type": "FeatureCollection",
"name": "Louisville_Metro_KY_County_Boundaries",
"crs": { "type": "name", "properties": { "name": "urn:ogc:def:crs:OGC:1.3:CRS84" } },
"features": [...]}
```

Where each item in `"features"` represent a county in the Louisville, KY metro area. This includes Jefferson County, where Louisville is, and several surrounding counties. Each `feature` or county object looks like this:

```json
{ "type": "Feature", 
  "properties": { "OBJECTID": 7, 
                  "CNTY_NAME": "JEFFERSON", 
                  "FIPS": "21111", 
                  "STATE_FIPS": "21", 
                  "CNTY_FIPS": "111", 
                  "SHAPEAREA": 11083783720.6446, 
                  "SHAPELEN": 513054.30366378697 }, 
  "geometry": { "type": "Polygon", 
                "coordinates": [ [ [ -85.575811868593064, 38.334545868470848 ], 
                                   [ -85.578070603648868, 38.335714204871564 ], 
                                   [ -85.579003771392806, 38.336193264774884 ],
                                    ... ] ] } }
```

Jefferson County is the one we are interested in here. The value for `"coordinates"` is a list of lists. Each interior list contains points defining a polygon that represents the boundary of the county. The points are listed as `[longitude, latitude]` pairs. The `coordinates` are really all we need, once we find the correct county object in the list of `features`.

In [3]:
METRO_BOUNDARIES = "data/raw/Louisville_Metro_KY_County_Boundaries.geojson"


def get_JEFFCO_data(path_to_data):
    data = get_data(path_to_data)
    counties = data['features']
    # Each feature represents a county boundary. Need to find the right one.
    # The county I am interested in is called Jefferson.

    for county in counties:
        if county['properties']['CNTY_NAME'] == 'JEFFERSON':
            JEFFCO = county
            break
    return JEFFCO

JEFFCO = get_JEFFCO_data(METRO_BOUNDARIES)

from math import sqrt

sqrt(JEFFCO['properties']['SHAPEAREA'])/5280


19.939308777281514

In [4]:

def get_JEFFCO_box(JEFFCO):
    points = JEFFCO['geometry']['coordinates'][0]
                                            # Sometimes Polygon geometry can consist of more than one shape.
                                            # In this case, the county boundary is just one shape.
    return pd.Series(sort_long_lat(points), name='official boundary')

boundaries = pd.DataFrame(get_JEFFCO_box(JEFFCO))
boundaries

,official boundary
west_longitude,-85.947127
east_longitude,-85.404922
south_latitude,37.997125
north_latitude,38.380238


In [5]:
# Both centerlines and intersection GeoJSON data contain an array of features, each of which has properties and geometry
# We are only concerned about the geometry, which has two properties, `type` and `coordinates`. `type` is not particularly helpful here. 

def get_geometries(json_data):
    """Pull all geometry objects out of GeoJSON, ignoring other data."""
    for feature in json_data['features']:
        yield feature['geometry']['coordinates']

In [6]:
# Get bounding box for centerlines
CENTERLINES = "data/raw/centerlines/Jefferson_County_KY_Street_Centerlines.geojson"

centerline_geometries = get_geometries(get_data(CENTERLINES))
centerline_points = (point for geometry in centerline_geometries for point in geometry)

boundaries['centerlines'] = sort_long_lat(centerline_points)
#boundaries

In [7]:
# Get bounding box for intersections

# Add bounding box from intersection metadata

INT_LL_BOX = {"west_longitude": -85.945347, "east_longitude": -85.344499,
              "north_latitude": 38.378034, "south_latitude": 38.005894}

boundaries['intersection metadata'] = INT_LL_BOX
#boundaries

In [8]:
# Derive box from intersection data

INTERSECTIONS = "data/raw/intersections/Jefferson_County_KY_Street_Intersections.geojson"
intersection_data = get_data(INTERSECTIONS)
intersection_points = get_geometries(intersection_data)

boundaries['intersections derived'] = sort_long_lat(intersection_points)
#boundaries


In [9]:
#Extent in the item's coordinate system 
west_longitude_co = 1154395.500000
east_longitude_co = 1325086.990000
south_latitude_co = 188677.437500
north_latitude_co = 321629.781250
#* Extent contains the resource Yes


def get_xy(data):
    for feature in data['features']:
        properties = feature['properties']
        x = properties['X_COORD']
        y= properties['Y_COORD']
        yield x, y

grid_box = sort_long_lat(get_xy(intersection_data)) # identical to stated extent
display(grid_box)


{'west_longitude': 1154395.5,
 'east_longitude': 1325086.99,
 'south_latitude': 188677.4375,
 'north_latitude': 321723.8175}

In [10]:

grid_span_x = abs(east_longitude_co - west_longitude_co)
grid_span_y = abs(north_latitude_co - south_latitude_co)

grid_span_y/5280 # == 25.180368134469695
grid_span_x/5280 # == 32.32793371212121

# convert coordinates from LOJIC CRS to (longitude, latitude)
# LOJIC projection: ESRI:102679
# NAD_1983_StatePlane_Kentucky_North_FIPS_1601_Feet

# Standard long, lat: epsg:4326
KY_grid_north_CRS = CRS("ESRI:102679")
standard_CRS = CRS("epsg:4326")

LL_to_KY_grid = Transformer.from_crs(crs_from=standard_CRS, crs_to=KY_grid_north_CRS, always_xy=True)
KY_grid_to_LL = Transformer.from_crs(crs_from=KY_grid_north_CRS, crs_to=standard_CRS, always_xy=True)

def xy_points():
    for x in (west_longitude_co, east_longitude_co):
        for y in (north_latitude_co, south_latitude_co):
            yield KY_grid_to_LL.transform(x,y)

SGC = sort_long_lat(xy_points())
boundaries['state_grid_conversion'] = SGC
boundaries

SG = pd.DataFrame(boundaries.state_grid_conversion)
SG['state_grid'] = {
    'west_longitude': west_longitude_co,
    'east_longitude': east_longitude_co,
    'north_latitude': north_latitude_co,
    'south_latitude': south_latitude_co}

long_span = haversine_distance_mi((SGC['west_longitude'], SGC['north_latitude']),
                      (SGC['east_longitude'], SGC['north_latitude']))
lat_span = haversine_distance_mi((SGC['west_longitude'], SGC['north_latitude']),
                      (SGC['west_longitude'], SGC['south_latitude']))

long_span, lat_span

NameError: name 'haversine_distance_mi' is not defined

In [ ]:
# compare boundary values to find the largest bounding box that covers all the data

BT = boundaries.T

comparisons = pd.Series({'west_longitude': BT.west_longitude.min(), 'east_longitude': BT.east_longitude.max(),
               'south_latitude': BT.south_latitude.min(), 'north_latitude': BT.north_latitude.max()},
               name = 'comparison')

pd.concat((boundaries, comparisons), axis=1)

,official boundary,centerlines,intersection metadata,intersections derived,state_grid_conversion,comparison
west_longitude,-85.947127,-85.942405,-85.945347,-85.936859,-85.945347,-85.947127
east_longitude,-85.404922,-85.344362,-85.344499,-85.347451,-85.344499,-85.344362
south_latitude,37.997125,38.000584,38.005894,38.005901,38.005894,37.997125
north_latitude,38.380238,38.377076,38.378034,38.375609,38.378034,38.380238


### Conclusion:

Official county boundary covers almost everything, except for `east_longitude` (maximum longitude), where some of the centerlines must have points east of the official county boundary. I also know that some of the intersections lie outside of Jefferson county.

The largest bounding box is represented by the `comparison` column in the last dataframe. This bounding box covers all the data across all the different input files. 

In [ ]:
display(comparisons)

longitude_delta = comparisons.east_longitude - comparisons.west_longitude
latitude_delta = comparisons.north_latitude - comparisons.south_latitude
longitude_delta, latitude_delta


west_longitude   -85.947127
east_longitude   -85.344362
south_latitude    37.997125
north_latitude    38.380238
Name: comparison, dtype: float64

(np.float64(0.6027649559007244), np.float64(0.3831129445748118))

In [ ]:
from common.county_geometry import *

east_point = (east_longitude, north_latitude)
west_point = (west_longitude, north_latitude)

long_mi = haversine_distance_mi(east_point, west_point)
long_km = haversine_distance_km(east_point, west_point)

display(f"longitude span = {long_mi} miles == {long_km} kilometers.")


north_point = (west_longitude, north_latitude)
south_point = (west_longitude, south_latitude)

lat_mi = haversine_distance_mi(south_point, north_point)
lat_km = haversine_distance_km(south_point, north_point)

display(f'latitude span = {lat_mi} miles == {lat_km} kilometers.')

'longitude span = 32.62464358670343 miles == 52.50786292126914 kilometers.'

'latitude span = 26.45211953861136 miles == 42.57346943941823 kilometers.'

In [ ]:
# TODO make this work
# TODO convert centerlines geometry -> first, last -> state grid for distance -> ft distance



# def convert_state_grid_coordinates(df):
#     # get projection info, create transformer
#     KY_grid_CRS = CRS("ESRI:102679")
#     long_lat_CRS = CRS("epsg:4326")
#     CRS_transformer = Transformer.from_crs(crs_from=KY_grid_CRS, crs_to=long_lat_CRS, always_xy=True)

#     # apply transformer to X, Y coordinates; add result to dataframe
#     conversion = df['X_COORD'].combine(df['Y_COORD'], CRS_transformer.transform)
#     return conversion

#     # drop now unneeded X, Y coordinate columns and return dataframe
#     return df.drop(["X_COORD", "Y_COORD"], axis=1)

# state_grid_conversion = convert_state_grid_coordinates(df2)
# state_grid_conversion

# df3 = df2.drop(["X_COORD", "Y_COORD"], axis=1)
# df3['state_grid_GEO'] = state_grid_conversion
# #df3
    
  

In [ ]:
# # county box
# west_longitude = -85.94712712079293
# east_longitude = -85.3443621648922
# south_latitude = 37.99712528351634
# north_latitude = 38.38023822809115

# delta_long = abs(east_longitude - west_longitude)
# delta_lat = abs(north_latitude - south_latitude)

# # # old point distance code that worked pretyt well


# long_dist = 30
# lat_dist = 26

# def n(point):
#     long, lat = point
#     v = ((long - west_longitude) / delta_long), ((lat - south_latitude) / delta_lat)
#     return np.array(v)

# def d(point):
#     long, lat = point
#     long = (long*long_dist)**2
#     lat = (lat*lat_dist)**2
#     return np.sqrt(long + lat)

# oft = (long_lat_coordinates.apply(n) - intersections.GEOMETRY.apply(n)).apply(d)*5480

# def dd(point):
#     long, lat = point
#     return (long/delta_long), (lat/delta_lat)

# ft = (long_lat_coordinates - intersections.GEOMETRY).apply(dd).apply(d)*5280

# ft.describe()
# ft[ft>=10]

# oft[oft>=100]

# ... stage before reorganizing column names

KY_grid_CRS = CRS("ESRI:102679")
long_lat_CRS = CRS("epsg:4326")

long_lat_to_grid = Transformer.from_crs(crs_from=long_lat_CRS, crs_to=KY_grid_CRS, always_xy=True)
ll2g_T = long_lat_to_grid.transform



# we have...
state_grid_conversion = state_grid_conversion.apply(np.array)
coordinates = df3.GEOMETRY.apply(np.array)

cc = (coordinates - state_grid_conversion)
cc.apply(np.linalg.norm).sort_values(ascending=True)

from common.county_geometry import delta_longitude, longitude_span_mi, delta_latitude, latitude_span_mi
from common.county_geometry import haversine_distance_mi as hav
from common.county_geometry import west_longitude as min_longitude
from common.county_geometry import south_latitude as min_latitude
from common.county_geometry import central_angle, earth_radius_mi, earth_radius_km, km_to_miles
from operator import itemgetter


pd.concat((
coordinates.combine(state_grid_conversion, hav),
cc.apply(np.linalg.norm)), axis=1)

def g(point):
    return hav((0,0), point)

pd.concat((
coordinates.combine(state_grid_conversion, hav),
cc.apply(np.linalg.norm),
cc.apply(g)), axis=1)


##### Info about X_COORD YCOORD system

via: https://www.lojic.org/data/projection-information

For this system, the Commonwealth shall be divided into a north zone and a south zone. The north zone shall be a Lambert conformal conic projection of the North American Datum of 1983, having standard parallels at north latitudes 37 degrees, 58 minutes, and 38 degrees, 58 minutes along which parallels the scale shall be exact. The origin of coordinates shall be at the intersection of the meridian 84 degrees, 15 minutes west of Greenwich, and the parallel 37 degrees, 30 minutes north latitude. This origin shall be given the coordinates: N=0, E=500,000.000 meters. The south zone shall be a Lambert conformal conic projection of the North American Datum of 1983, having standard parallels at north latitudes 36 degrees, 44 minutes, and 37 degrees, 56 minutes along which parallels the scale shall be exact. The origin of coordinates shall be at the intersection of the meridian 85 degrees, 45 minutes west of Greenwich, and the parallel 36 degrees, 20 minutes north latitude. This origin shall be given the coordinates: N=500,000.000, E=500,000.000 meters. The southern edge of the following counties shall delineate the boundary between the north zone and the south zone: Bullitt, Spencer, Anderson, Woodford, Jessamine, Fayette, Clark, Montgomery, Menifee, Morgan, and Lawrence.

One U. S. survey foot equals (1200)/(3937) meter. For conversion of meters to U. S. survey feet, multiply the meters by 3.28083333333 to twelve (12) significant figures. When converting from meters to feet, the conversion factor defined by the U. S. survey foot shall be used.


The plane coordinate values for a point on the earth's surface, used to express the geographic position or location of the point in the appropriate zone of this system, shall consist of two (2) distances expressed in U. S. survey feet and decimals of a foot when using the Kentucky Coordinate System of 1983. For the Kentucky Coordinate System of 1983, one (1) of the distances, to be known as the "northing" or "N", shall give the position in a north/south direction. The other, to be known as the "easting" or "E" shall give the position in an east/west direction. These coordinates shall be made to depend upon and conform to plane rectangular coordinates values for the monumented points of the North American National Geodetic Horizontal Network as published by the National Ocean Service/National Geodetic Survey, and whose plane coordinates have been computed on the systems established by the National Ocean Service/National Geodetic Survey. Any such station may be used for establishing a survey connection to the Kentucky Coordinate System of 1983.

In [ ]:
xy = df2[['X_COORD', 'Y_COORD']]

def convert_long_lat_to_grid(point):
    return ll2g_T(*point)

llconv = coordinates.apply(convert_long_lat_to_grid)
llconv = llconv.transform({"X":itemgetter(0), "Y":itemgetter(1)})

gf = pd.concat((xy, llconv), axis=1)
gf['x_diff'] = x_diff = (xy.X_COORD - llconv.X)
gf['y_diff'] = y_diff = (xy.Y_COORD - llconv.Y)


d = np.sqrt(x_diff**2 + y_diff**2)


NameError: name 'df2' is not defined

In [ ]:

from common.county_geometry import earth_radius_km_2

erm = earth_radius_mi
dd = (state_grid_conversion.combine(coordinates, central_angle)*erm*5280) - d

dd[dd > .01]

from common import county_geometry

hav = county_geometry.haversine_distance_mi

dd = df3.GEOMETRY.combine(df3.state_grid_GEO, hav)
dd
